# 6.34 — Hardware & Mixed Precision

Hardware-aware training is the numerical side of deep learning: we choose array shapes that keep matrix units busy, store bulky activations in cheaper formats, and still protect the tiny gradient updates that make learning work. In this lesson, you will simulate FP16 and bfloat16 behavior with NumPy, inspect loss scaling and unscaling, and see why mixed precision usually means **16-bit compute plus a 32-bit master copy** rather than blindly making everything low precision.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build hardware and mixed precision one idea at a time. Run each cell in order and read the printed intermediate values — every piece of arithmetic is small enough to inspect, and every approximation is tied back to why training either stays stable or breaks. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vectorized arithmetic, and simulated low-precision formats.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for synthetic activations and gradients.

def to_fp16_w(x):  # simulate IEEE float16 storage followed by float32 compute.
    return np.asarray(x, dtype=np.float16).astype(np.float32)

def to_bf16_w(x):  # simulate bfloat16 by truncating the low 16 mantissa bits of float32.
    x32 = np.asarray(x, dtype=np.float32)
    bits = x32.view(np.uint32)
    rounded = bits + np.uint32(0x00008000)
    bf_bits = rounded & np.uint32(0xFFFF0000)
    return bf_bits.view(np.float32)

### 1. Hardware wants big matrix work, not one scalar at a time

A GPU or TPU is fast because many multiply-add units work at once. The mathematical operation is still the dot product, but batching turns many small independent dot products into one matrix multiplication. That is why deep learning code is written as arrays: it lets hardware reuse the same instruction pattern across many examples and keeps expensive matrix units busy.

In [ ]:
X_w = np.array([[1.5, -0.5], [0.2, 1.0], [-1.0, 0.3], [2.0, 0.1]], dtype=np.float32)  # four examples.
W_w = np.array([[1.8], [-0.1]], dtype=np.float32)  # one output unit with two input weights.
b_w = np.array([0.7], dtype=np.float32)  # bias from the lesson arithmetic.
Z_w = X_w @ W_w + b_w  # batched affine map: every row gets the same weights.
A_w = np.maximum(Z_w, 0.0)  # ReLU gate keeps positive signal and clips negative signal.

print("first affine:", round(float(Z_w[0, 0]), 3))
print("first gated:", round(float(A_w[0, 0]), 3))

assert round(float(Z_w[0, 0]), 3) == 3.45

▶ What you'll see: the first example reproduces `1.8·1.5 + (-0.1)·(-0.5) + 0.7 = 3.45`, while the batch computes four rows at once.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(np.arange(len(Z_w)), Z_w.ravel(), color="steelblue", label="affine")
plt.bar(np.arange(len(A_w)), A_w.ravel(), alpha=0.45, color="seagreen", label="ReLU")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("1: batched affine signals")
plt.xlabel("example in batch")
plt.ylabel("signal")
plt.legend()
plt.show()

▶ What you'll see: positive affine values pass through unchanged; any negative value would be clipped to zero by the gate.

*Why it's done this way:* the dot product $x^\top w+b$ is the scalar building block, but the matrix product $XW+b$ is the hardware-friendly version. It expresses the same math while exposing parallel rows and columns, so the accelerator can do many multiply-adds per memory fetch instead of waiting on one scalar loop.

### 2. FP16 and bfloat16 make different rounding promises

Mixed precision is not just “fewer bits.” FP16 keeps more fraction precision near 1 but has a smaller exponent range; bfloat16 keeps the float32-like exponent range but fewer fraction bits. That means FP16 can represent small local detail better, while bfloat16 survives very large or very tiny magnitudes more gracefully.

In [ ]:
values_w = np.array([1.0, 1.001, 3.45, 1e-5, 1e5], dtype=np.float32)  # span ordinary and extreme scales.
fp16_w = to_fp16_w(values_w)  # simulate float16 storage.
bf16_w = to_bf16_w(values_w)  # simulate bfloat16 storage.

print("float32:", values_w)
print("fp16   :", fp16_w)
print("bf16   :", bf16_w)

assert round(float(fp16_w[2]), 3) == 3.449

▶ What you'll see: FP16 tracks `3.45` closely but overflows `1e5` to `inf`; bfloat16 keeps `1e5` finite but rounds ordinary numbers more coarsely.

In [ ]:
abs_err_w = np.abs(np.vstack([fp16_w, bf16_w]) - values_w)
plt.figure(figsize=(5, 3))
plt.semilogy(abs_err_w.T + 1e-12, marker="o")
plt.xticks(range(len(values_w)), ["1", "1.001", "3.45", "1e-5", "1e5"])
plt.title("2: absolute rounding error by format")
plt.ylabel("absolute error (log scale)")
plt.legend(["fp16", "bf16"])
plt.show()

▶ What you'll see: the error pattern changes by scale; no 16-bit format is uniformly best for every number.

*Why it's done this way:* training has both moderate activations and extreme gradients. FP16's tighter mantissa can be useful near normal activation scales, but bfloat16's wider exponent reduces overflow and underflow surprises. Hardware-aware training chooses the format whose failure mode is easiest to manage.

### 3. Loss scaling protects small gradients before FP16 storage

Tiny gradients can underflow when stored in FP16. Loss scaling multiplies the loss, so every gradient is multiplied by the same scale before low-precision storage; after storage, we unscale by dividing by that scale. The update direction is unchanged in exact arithmetic, but the representable stored gradient is no longer rounded to zero.

In [ ]:
grad_w = np.array([2.1, 1e-8, -3e-8], dtype=np.float32)  # one ordinary gradient and two tiny gradients.
scale_w = 4096.0  # common power-of-two style scale factor.
stored_direct_w = to_fp16_w(grad_w)  # store gradients directly in fp16.
stored_scaled_w = to_fp16_w(grad_w * scale_w) / scale_w  # scale, store, then unscale.

print("direct fp16:", stored_direct_w)
print("scaled/unscaled:", stored_scaled_w)

assert stored_direct_w[1] == 0.0 and stored_scaled_w[1] > 0.0

▶ What you'll see: the `1e-8` gradient disappears when stored directly, but survives when scaled before storage.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["true tiny", "direct fp16", "scaled→unscaled"], [grad_w[1], stored_direct_w[1], stored_scaled_w[1]], color=["gray", "crimson", "seagreen"])
plt.title("3: loss scaling rescues a tiny gradient")
plt.ylabel("gradient value")
plt.show()

▶ What you'll see: the direct FP16 bar is zero, while the scaled-and-unscaled bar remains close to the original tiny gradient.

*Why it's done this way:* multiplying the loss by $S$ multiplies each gradient by $S$ because differentiation is linear. Dividing by $S$ after storage restores the original gradient scale, so the math target is the same but the intermediate number lands inside FP16's representable range.

### 4. The 32-bit master copy preserves small parameter movement

The core mixed-precision update is $\theta_{32}\leftarrow\theta_{32}-\eta\,\mathrm{unscale}(g_{16})$. Activations and gradients may use 16-bit storage, but the parameter that accumulates many tiny nudges should remain 32-bit. If the weight itself lives only in FP16, repeated small updates can round away and learning stalls.

In [ ]:
theta32_w = np.float32(2.0)  # master parameter.
theta16_w = np.float16(2.0)  # low-precision-only parameter for comparison.
eta_w = np.float32(0.09)
g_w = np.float32(2.1)
update_w = eta_w * g_w
new32_w = np.float32(theta32_w - update_w)
new16_w = np.float16(theta16_w - np.float16(update_w))

print("update:", round(float(update_w), 3))
print("fp32 master:", round(float(new32_w), 3), "fp16-only:", float(new16_w))

assert round(float(new32_w), 3) == 1.811

▶ What you'll see: the lesson update `2.000 - 0.090·2.100 = 1.811` is preserved by the FP32 master copy.

In [ ]:
theta_master_w = np.float32(1.0)
theta_low_w = np.float16(1.0)
small_step_w = np.float32(1e-4)
trace_master_w, trace_low_w = [], []
for _ in range(20):
    theta_master_w = np.float32(theta_master_w - small_step_w)
    theta_low_w = np.float16(theta_low_w - np.float16(small_step_w))
    trace_master_w.append(float(theta_master_w))
    trace_low_w.append(float(theta_low_w))

print("after 20 small steps:", round(trace_master_w[-1], 6), float(trace_low_w[-1]))

assert round(trace_master_w[-1], 3) == 0.998

▶ What you'll see: the FP32 master moves by exactly about `0.002`, while the FP16-only value barely moves because each small subtraction is below the local spacing.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(trace_master_w, label="fp32 master", color="seagreen")
plt.plot(trace_low_w, label="fp16-only", color="crimson")
plt.title("4: small updates accumulate only in the master copy")
plt.xlabel("step")
plt.ylabel("parameter value")
plt.legend()
plt.show()

▶ What you'll see: the green line steadily descends; the red line is nearly flat.

*Why it's done this way:* optimization is the sum of many small corrections. If the storage format cannot represent one correction at the current parameter scale, doing the subtraction in that format erases the signal. A 32-bit master copy keeps the accumulator precise while still allowing cheaper 16-bit forward/backward arrays.

### 5. Normalization and softmax are scale-management tools

Deep networks turn scores into comparisons and then gradients. If a score is far from the batch scale, normalization tells us how unusual it is; if two scores compete, softmax turns their difference into a probability. Both operations are scale-sensitive, so mixed precision code must keep them numerically stable.

In [ ]:
score_w = np.float32(3.45)
base_w = np.float32(0.4)
prob_w = np.exp(score_w) / (np.exp(score_w) + np.exp(base_w))
normed_w = (score_w - 1.0) / np.sqrt(0.25 + 0.00001)

print("softmax probability:", round(float(prob_w), 3))
print("normalized value:", round(float(normed_w), 3))

assert round(float(prob_w), 3) == 0.955
assert round(float(normed_w), 3) == 4.9

▶ What you'll see: the score `3.45` beats baseline `0.4` with probability about `0.955`, and it is `4.9` standard deviations above a mean-1 variance-0.25 normalization.

In [ ]:
logits_w = np.array([3.45, 0.4], dtype=np.float32)
shifted_w = logits_w - np.max(logits_w)  # stable softmax shift.
softmax_w = np.exp(shifted_w) / np.sum(np.exp(shifted_w))
plt.figure(figsize=(4, 3))
plt.bar(["score", "baseline"], softmax_w, color=["steelblue", "gray"])
plt.title("5: stable softmax after max-shift")
plt.ylabel("probability")
plt.show()

▶ What you'll see: the score gets nearly all probability mass, but the computation avoids unnecessarily large exponentials.

*Why it's done this way:* subtracting the max leaves all softmax probabilities unchanged because the same constant cancels from numerator and denominator. That algebraic trick matters in low precision: it keeps exponentials in a safe range while preserving the comparison the model actually needs.

### 6. Memory bandwidth is part of the math budget

A tiny activation block with 4 vectors of length 128 in float32 uses $4\cdot128\cdot4/1024=2$ KB. Halving bytes per value does not change the formula, but it changes how many activations fit in memory and how much data must move between memory and compute units. Hardware-aware training treats memory as a first-class constraint.

In [ ]:
batch_w, width_w = 4, 128
bytes32_w = batch_w * width_w * 4
bytes16_w = batch_w * width_w * 2

print("float32 KB:", bytes32_w / 1024)
print("float16/bfloat16 KB:", bytes16_w / 1024)

assert bytes32_w / 1024 == 2.0
assert bytes16_w / 1024 == 1.0

▶ What you'll see: this small block drops from `2.0` KB to `1.0` KB when activations are stored with 2 bytes each.

In [ ]:
layers_w = np.array([1, 4, 16, 64])
mem32_w = layers_w * bytes32_w / 1024
mem16_w = layers_w * bytes16_w / 1024
plt.figure(figsize=(5, 3))
plt.plot(layers_w, mem32_w, marker="o", label="float32 activations")
plt.plot(layers_w, mem16_w, marker="o", label="16-bit activations")
plt.title("6: activation memory scales with layers")
plt.xlabel("number of stored activation blocks")
plt.ylabel("KB")
plt.legend()
plt.show()

▶ What you'll see: both lines grow linearly with depth, and the 16-bit line stays exactly half as high.

*Why it's done this way:* memory cost is `number of values × bytes per value`. Mixed precision reduces the byte multiplier for large tensors, which can enable larger batches or models; the FP32 master copy is kept only where accumulated numerical accuracy matters most.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Each hardware toy uses a handful of numbers
> to isolate batching, low-precision rounding, loss scaling, FP32 master updates, stable comparisons,
> or memory accounting. Every block prints intermediates, draws one visual check, and asserts the result.

### ✍️ Toy 1 · Batched affine work feeds ReLU

A matrix multiply runs the same affine map on four examples at once, then ReLU clips negative rows.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)                 # seeded for reproducibility.
t1_X = np.array([[1.0, -1.0], [2.0, 0.5], [-1.0, 1.0], [0.0, 2.0]], dtype=np.float32)
t1_W = np.array([[1.5], [-0.5]], dtype=np.float32)
t1_b = np.array([0.25], dtype=np.float32)

print("batch X:", t1_X.tolist())                                      # -> [[1.0, -1.0], [2.0, 0.5], [-1.0, 1.0], [0.0, 2.0]]
print("weights:", t1_W.ravel().tolist(), "bias:", t1_b.tolist())     # -> [1.5, -0.5] bias [0.25]

t1_Z = t1_X @ t1_W + t1_b

print("affine outputs:", np.round(t1_Z.ravel(), 3).tolist())          # -> [2.25, 3.0, -1.75, -0.75]

t1_A = np.maximum(t1_Z, 0.0)

print("ReLU outputs:", np.round(t1_A.ravel(), 3).tolist())            # -> [2.25, 3.0, 0.0, 0.0]

assert np.allclose(t1_A.ravel(), [2.25, 3.0, 0.0, 0.0])

plt.figure(figsize=(4.8, 3.0))
t1_idx = np.arange(t1_Z.size)
plt.bar(t1_idx - 0.18, t1_Z.ravel(), width=0.36, label="affine", color="steelblue")
plt.bar(t1_idx + 0.18, t1_A.ravel(), width=0.36, label="ReLU", color="seagreen")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("example")
plt.ylabel("signal")
plt.title("Toy 1 · batched affine plus gate")
plt.legend()
plt.show()

▶ What you'll see: four rows are computed together, and the two negative affine outputs become zeros.

### ✍️ Toy 2 · FP16 and bfloat16 round different scales differently

FP16 has finer local detail near ordinary values, while bfloat16 preserves the large exponent range.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)                 # seeded for reproducibility.
def t2_to_bf16(t2_x):
    t2_x32 = np.asarray(t2_x, dtype=np.float32)
    t2_bits = t2_x32.view(np.uint32)
    t2_rounded = t2_bits + np.uint32(0x00008000)
    t2_bf_bits = t2_rounded & np.uint32(0xFFFF0000)
    return t2_bf_bits.view(np.float32)

t2_values = np.array([1.0, 1.001, 3.25, 1e-5, 1e5, -2.5], dtype=np.float32)

print("float32 values:", t2_values.tolist())                         # -> [1.0, 1.0010000467300415, 3.25, 9.999999747378752e-06, 100000.0, -2.5]

t2_fp16 = np.asarray(t2_values, dtype=np.float16).astype(np.float32)

print("fp16 stored:", t2_fp16.tolist())                              # -> [1.0, 1.0009765625, 3.25, 1.0013580322265625e-05, inf, -2.5]

t2_bf16 = t2_to_bf16(t2_values)

print("bf16 stored:", t2_bf16.tolist())                              # -> [1.0, 1.0, 3.25, 1.0013580322265625e-05, 99840.0, -2.5]

t2_errors = np.abs(np.vstack([t2_fp16, t2_bf16]) - t2_values)

print("absolute errors:", np.round(t2_errors, 6).tolist())           # -> [[0.0, 2.3e-05, 0.0, 0.0, inf, 0.0], [0.0, 0.001, 0.0, 0.0, 160.0, 0.0]]

assert np.isinf(t2_fp16[4])
assert np.isfinite(t2_bf16[4])

plt.figure(figsize=(5.0, 3.0))
plt.semilogy(t2_errors.T + 1e-12, marker="o")
plt.xticks(np.arange(t2_values.size), ["1", "1.001", "3.25", "1e-5", "1e5", "-2.5"], rotation=20)
plt.ylabel("absolute error")
plt.title("Toy 2 · rounding error depends on format")
plt.legend(["fp16", "bf16"])
plt.show()

▶ What you'll see: FP16 overflows `1e5`, while bfloat16 keeps it finite but rounds `1.001` more coarsely.

### ✍️ Toy 3 · Loss scaling rescues tiny gradients

Scaling before FP16 storage moves tiny gradients into the representable range; unscaling restores their original scale.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)                 # seeded for reproducibility.
t3_grad = np.array([2e-3, 1e-8, -3e-8, 5e-5, -1e-5, 0.1], dtype=np.float32)
t3_scale = 4096.0

print("true gradients:", t3_grad.tolist())                           # -> [0.0020000000949949026, 9.99999993922529e-09, -2.999999892949745e-08, 4.999999873689376e-05, -9.999999747378752e-06, 0.10000000149011612]
print("loss scale:", t3_scale)                                       # -> 4096.0

t3_direct = np.asarray(t3_grad, dtype=np.float16).astype(np.float32)

print("direct fp16:", t3_direct.tolist())                            # -> [0.0020008087158203125, 0.0, -5.960464477539063e-08, 5.0008296966552734e-05, -1.0013580322265625e-05, 0.0999755859375]

t3_scaled = np.asarray(t3_grad * t3_scale, dtype=np.float16).astype(np.float32) / t3_scale

print("scaled then unscaled:", t3_scaled.tolist())                   # -> [0.0020008087158203125, 9.997165761888027e-09, -3.000604920089245e-08, 5.0008296966552734e-05, -9.998679161071777e-06, 0.0999755859375]

assert t3_direct[1] == 0.0
assert t3_scaled[1] > 0.0

plt.figure(figsize=(4.8, 3.0))
plt.bar(["true", "direct", "scaled"], [t3_grad[1], t3_direct[1], t3_scaled[1]], color=["gray", "crimson", "seagreen"])
plt.ylabel("gradient")
plt.title("Toy 3 · scaling protects 1e-8")
plt.show()

▶ What you'll see: the direct FP16 path rounds the `1e-8` gradient to zero, but the scaled path keeps it positive.

### ✍️ Toy 4 · FP32 master weights accumulate small updates

A low-precision parameter can ignore tiny steps near `1.0`, while an FP32 master copy keeps accumulating them.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)                 # seeded for reproducibility.
t4_master = np.float32(1.0)
t4_low = np.float16(1.0)
t4_step = np.float32(1e-4)

print("initial master:", float(t4_master))                            # -> 1.0
print("initial fp16-only:", float(t4_low))                            # -> 1.0
print("step size:", float(t4_step))                                  # -> 9.999999747378752e-05

t4_master_trace = []
t4_low_trace = []
for t4_i in range(8):
    t4_master = np.float32(t4_master - t4_step)
    t4_low = np.float16(t4_low - np.float16(t4_step))
    t4_master_trace.append(float(t4_master))
    t4_low_trace.append(float(t4_low))

print("master trace:", np.round(t4_master_trace, 6).tolist())        # -> [0.9999, 0.9998, 0.9997, 0.9996, 0.9995, 0.9994, 0.9993, 0.9992]
print("fp16-only trace:", t4_low_trace)                              # -> [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

assert round(t4_master_trace[-1], 4) == 0.9992
assert t4_low_trace[-1] == 1.0

plt.figure(figsize=(4.8, 3.0))
plt.plot(t4_master_trace, marker="o", label="fp32 master", color="seagreen")
plt.plot(t4_low_trace, marker="x", label="fp16-only", color="crimson")
plt.xlabel("update")
plt.ylabel("parameter value")
plt.title("Toy 4 · tiny updates need a master copy")
plt.legend()
plt.show()

▶ What you'll see: the FP32 master descends by `0.0008`, while the FP16-only value stays flat.

### ✍️ Toy 5 · Stable softmax and normalization manage scale

Subtracting the maximum preserves softmax probabilities, and normalization turns raw logits into standardized deviations.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)                 # seeded for reproducibility.
t5_logits = np.array([12.0, 9.0, 3.0, -2.0, 0.0, 6.0])

print("logits:", t5_logits.tolist())                                  # -> [12.0, 9.0, 3.0, -2.0, 0.0, 6.0]

t5_shifted = t5_logits - t5_logits.max()

print("shifted logits:", t5_shifted.tolist())                        # -> [0.0, -3.0, -9.0, -14.0, -12.0, -6.0]

t5_exp = np.exp(t5_shifted)

print("shifted exponentials:", np.round(t5_exp, 6).tolist())         # -> [1.0, 0.049787, 0.000123, 1e-06, 6e-06, 0.002479]

t5_probs = t5_exp / t5_exp.sum()

print("softmax probabilities:", np.round(t5_probs, 4).tolist())      # -> [0.9502, 0.0473, 0.0001, 0.0, 0.0, 0.0024]

t5_mean = t5_logits.mean()

print("mean:", round(float(t5_mean), 3))                             # -> 4.667

t5_std = t5_logits.std()

print("standard deviation:", round(float(t5_std), 3))                # -> 4.888

t5_normed = (t5_logits - t5_mean) / t5_std

print("normalized logits:", np.round(t5_normed, 3).tolist())         # -> [1.5, 0.887, -0.341, -1.364, -0.955, 0.273]

assert abs(float(t5_probs.sum()) - 1.0) < 1e-12
assert int(np.argmax(t5_probs)) == 0

plt.figure(figsize=(4.8, 3.0))
plt.bar(np.arange(t5_probs.size), t5_probs, color="steelblue")
plt.xlabel("logit index")
plt.ylabel("probability")
plt.title("Toy 5 · max-shifted softmax")
plt.show()

▶ What you'll see: the largest logit owns most probability, but the shifted exponentials remain numerically tame.

### ✍️ Toy 6 · Activation memory scales with bytes per value

The same activation block costs twice as many bytes in float32 as in a 16-bit format, and that gap grows with layers.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)                 # seeded for reproducibility.
t6_layers = np.arange(1, 7)
t6_batch = 3
t6_width = 4

print("layers:", t6_layers.tolist())                                  # -> [1, 2, 3, 4, 5, 6]
print("batch:", t6_batch, "width:", t6_width)                       # -> 3 width 4

t6_bytes32_block = t6_batch * t6_width * 4
t6_bytes16_block = t6_batch * t6_width * 2

print("float32 bytes per block:", t6_bytes32_block)                  # -> 48
print("16-bit bytes per block:", t6_bytes16_block)                   # -> 24

t6_total32 = t6_layers * t6_bytes32_block

print("float32 totals:", t6_total32.tolist())                        # -> [48, 96, 144, 192, 240, 288]

t6_total16 = t6_layers * t6_bytes16_block

print("16-bit totals:", t6_total16.tolist())                         # -> [24, 48, 72, 96, 120, 144]

t6_ratio = t6_total32 / t6_total16

print("memory ratio:", t6_ratio.tolist())                            # -> [2.0, 2.0, 2.0, 2.0, 2.0, 2.0]

assert np.all(t6_ratio == 2.0)

plt.figure(figsize=(4.8, 3.0))
plt.plot(t6_layers, t6_total32, marker="o", label="float32", color="gray")
plt.plot(t6_layers, t6_total16, marker="o", label="16-bit", color="seagreen")
plt.xlabel("stored activation blocks")
plt.ylabel("bytes")
plt.title("Toy 6 · 16-bit cuts activation bytes in half")
plt.legend()
plt.show()

▶ What you'll see: both memory curves grow linearly, and the 16-bit line stays exactly half the float32 line.

## 🛠️ Setup

In [ ]:
import numpy as np  # Import NumPy for arrays, matrix multiplication, numerical formats, and deterministic simulations.
import matplotlib.pyplot as plt  # Import Matplotlib for the small diagnostic plots used throughout the lesson.
np.random.seed(0)  # Fix the global random seed so every stochastic example is reproducible.

def fp16(x):  # Simulate storing values in IEEE float16 and then converting back to float32 for inspection.
    return np.asarray(x, dtype=np.float16).astype(np.float32)  # Round through float16 while returning a NumPy float32 array.

def bf16(x):  # Simulate bfloat16 by rounding float32 values and zeroing the low 16 mantissa bits.
    x32 = np.asarray(x, dtype=np.float32)  # Work from float32 so the bit layout is predictable.
    bits = x32.view(np.uint32)  # Reinterpret the float32 bits as unsigned integers.
    rounded = bits + np.uint32(0x00008000)  # Add half of the truncated region for round-to-nearest behavior.
    return (rounded & np.uint32(0xFFFF0000)).view(np.float32)  # Keep sign, exponent, and top mantissa bits.

def softmax_stable(logits):  # Define a stable softmax helper for small score vectors.
    z = np.asarray(logits, dtype=np.float32)  # Convert logits to float32 before exponentials.
    z = z - np.max(z)  # Subtract the maximum because this leaves probabilities unchanged and prevents overflow.
    ez = np.exp(z)  # Exponentiate shifted logits.
    return ez / np.sum(ez)  # Normalize exponentials into probabilities.

def activation_kb(batch, width, bytes_per_value):  # Compute activation memory in kilobytes.
    return batch * width * bytes_per_value / 1024  # Values times bytes, converted from bytes to KB.

## 🟢 Basics (warm-up)

### Basic 1 — Batch the affine map

**Goal.** Compute the lesson's tiny affine signal as a batch operation, because hardware is efficient when many examples share one matrix multiply. We build it in 2 steps.

In [ ]:
X_b1 = np.array([[1.5, -0.5], [0.2, 1.0], [-1.0, 0.3], [2.0, 0.1]], dtype=np.float32)  # Store four two-feature examples.
W_b1 = np.array([[1.8], [-0.1]], dtype=np.float32)  # Store one output unit's weights.
b_b1 = np.array([0.7], dtype=np.float32)  # Store the bias term.
Z_b1 = X_b1 @ W_b1 + b_b1  # Compute all affine outputs at once.

print("affine outputs:", np.round(Z_b1.ravel(), 3))  # Inspect the batched scores.

assert round(float(Z_b1[0, 0]), 3) == 3.45  # Verify the lesson arithmetic.

▶ What you'll see: the first row is `3.45`, and the other rows are computed by the same batched matrix expression.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a compact bar chart.
plt.bar(np.arange(len(Z_b1)), Z_b1.ravel(), color="steelblue")  # Show one affine score per batch row.
plt.title("Basic 1: batched affine outputs")  # Title the plot.
plt.xlabel("batch row")  # Label examples.
plt.ylabel("xW+b")  # Label affine signal.
plt.show()  # Display the chart.

▶ What you'll see: each bar is one row's affine result from the same matrix multiplication.

👀 Takeaway: batching keeps the math identical while exposing parallel work to the hardware.

### Basic 2 — Apply a ReLU gate

**Goal.** Pass positive signals and clip negative signals, because deep networks reshape activations before the next layer uses them. We build it in 2 steps.

In [ ]:
signals_b2 = np.array([3.45, -0.2, 0.0, 1.3], dtype=np.float32)  # Define four pre-activation signals.
gated_b2 = np.maximum(signals_b2, 0.0)  # Apply ReLU elementwise.

print("before:", signals_b2)  # Inspect raw signals.
print("after :", gated_b2)  # Inspect gated activations.

assert round(float(gated_b2[0]), 2) == 3.45 and gated_b2[1] == 0.0  # Check pass-through and clipping.

▶ What you'll see: positive `3.45` stays positive, while `-0.2` becomes zero.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a before/after chart.
plt.plot(signals_b2, marker="o", label="before")  # Plot pre-activation values.
plt.plot(gated_b2, marker="o", label="after ReLU")  # Plot post-gate values.
plt.axhline(0, color="black", linewidth=0.8)  # Mark the clipping boundary.
plt.title("Basic 2: ReLU gate")  # Title the figure.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: the post-ReLU line never drops below zero.

👀 Takeaway: a gate is simple local arithmetic, but it changes which signals and gradients can pass forward.

### Basic 3 — Compute one SGD update

**Goal.** Move one scalar parameter with the lesson's learning rate and gradient, because training is many reliable nudges. We build it in 2 steps.

In [ ]:
theta_b3 = np.float32(2.0)  # Store the starting parameter.
eta_b3 = np.float32(0.09)  # Store the learning rate.
grad_b3 = np.float32(2.1)  # Store the gradient.
step_b3 = eta_b3 * grad_b3  # Compute the amount to subtract.

print("step size:", round(float(step_b3), 3))  # Inspect ηg.

▶ What you'll see: the scalar step size is `0.189`.

In [ ]:
new_theta_b3 = theta_b3 - step_b3  # Apply gradient descent.

print("new theta:", round(float(new_theta_b3), 3))  # Inspect the updated parameter.

assert round(float(new_theta_b3), 3) == 1.811  # Verify the lesson number.
plt.figure(figsize=(4, 3))  # Create a compact before/after plot.
plt.bar(["before", "after"], [theta_b3, new_theta_b3], color=["gray", "seagreen"])  # Show the parameter movement.
plt.title("Basic 3: one gradient step")  # Title the plot.
plt.ylabel("θ")  # Label parameter value.
plt.show()  # Display the chart.

▶ What you'll see: the parameter decreases from `2.000` to `1.811`.

👀 Takeaway: gradient descent uses a scaled gradient, not an arbitrary jump.

### Basic 4 — Compare FP32 and FP16 rounding

**Goal.** Round ordinary values through FP16, because mixed precision changes stored numbers before later math uses them. We build it in 2 steps.

In [ ]:
vals_b4 = np.array([1.0, 1.001, 3.45, 10.25], dtype=np.float32)  # Choose ordinary activation-scale values.
vals16_b4 = fp16(vals_b4)  # Store them as float16 and inspect as float32.

print("fp32:", vals_b4)  # Inspect original values.
print("fp16:", vals16_b4)  # Inspect rounded values.

assert round(float(vals16_b4[2]), 3) == 3.449  # Check the rounded 3.45 value.

▶ What you'll see: FP16 keeps values close but not exact; `3.45` becomes about `3.449`.

In [ ]:
err_b4 = np.abs(vals16_b4 - vals_b4)  # Compute absolute rounding errors.
plt.figure(figsize=(4, 3))  # Create an error plot.
plt.bar(["1", "1.001", "3.45", "10.25"], err_b4, color="orange")  # Show one rounding error per value.
plt.title("Basic 4: FP16 rounding error")  # Title the plot.
plt.ylabel("absolute error")  # Label the error axis.
plt.show()  # Display the chart.

▶ What you'll see: each bar is small but nonzero when the value is not exactly representable.

👀 Takeaway: low precision saves memory by accepting controlled rounding error.

### Basic 5 — Simulate bfloat16 rounding

**Goal.** Compare bfloat16-style rounding with FP16, because bfloat16 preserves range while using fewer fraction bits. We build it in 2 steps.

In [ ]:
vals_b5 = np.array([1.001, 3.45, 1e5], dtype=np.float32)  # Include both ordinary and large values.
f16_b5 = fp16(vals_b5)  # Round through FP16.
bf_b5 = bf16(vals_b5)  # Round through simulated bfloat16.

print("fp16:", f16_b5)  # Inspect FP16 results.
print("bf16:", bf_b5)  # Inspect bfloat16 results.

assert np.isinf(f16_b5[2]) and np.isfinite(bf_b5[2])  # FP16 overflows 1e5; bfloat16 does not.

▶ What you'll see: FP16 overflows `1e5`, while bfloat16 keeps a finite approximate value.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a comparison chart.
plt.bar(["fp16 1e5", "bf16 1e5"], [65504, bf_b5[2]], color=["crimson", "seagreen"])  # Use FP16 max as a finite stand-in for the overflow bar.
plt.title("Basic 5: range difference")  # Title the figure.
plt.ylabel("displayed finite value")  # Label the scale.
plt.show()  # Display the chart.

▶ What you'll see: the bfloat16 bar remains near `100000`, while FP16's finite range tops out much lower.

👀 Takeaway: bfloat16 trades mantissa detail for exponent range, which is often safer for deep learning scales.

### Basic 6 — Measure activation memory

**Goal.** Compute the memory for a small activation block, because hardware-aware training tracks bytes as carefully as formulas. We build it in 2 steps.

In [ ]:
batch_b6 = 4  # Use the lesson's four activation vectors.
width_b6 = 128  # Use the lesson's vector length.
kb32_b6 = activation_kb(batch_b6, width_b6, 4)  # Compute float32 memory.
kb16_b6 = activation_kb(batch_b6, width_b6, 2)  # Compute 16-bit memory.

print("float32 KB:", kb32_b6)  # Inspect 32-bit storage.
print("16-bit KB:", kb16_b6)  # Inspect 16-bit storage.

assert kb32_b6 == 2.0 and kb16_b6 == 1.0  # Verify concrete memory numbers.

▶ What you'll see: the same activation block uses `2.0` KB in float32 and `1.0` KB in 16-bit storage.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a memory comparison chart.
plt.bar(["float32", "16-bit"], [kb32_b6, kb16_b6], color=["gray", "seagreen"])  # Plot memory by format.
plt.title("Basic 6: activation memory")  # Title the chart.
plt.ylabel("KB")  # Label memory units.
plt.show()  # Display the chart.

▶ What you'll see: the 16-bit bar is exactly half the float32 bar.

👀 Takeaway: mixed precision often wins by reducing the biggest tensor storage and bandwidth costs.

### Basic 7 — Normalize a signal

**Goal.** Convert the lesson score into a standardized value, because scale affects gradient and activation behavior. We build it in 2 steps.

In [ ]:
score_b7 = np.float32(3.45)  # Use the lesson score.
mean_b7 = np.float32(1.0)  # Use the lesson normalization mean.
var_b7 = np.float32(0.25)  # Use the lesson normalization variance.
eps_b7 = np.float32(1e-5)  # Add epsilon to avoid division by zero.
normed_b7 = (score_b7 - mean_b7) / np.sqrt(var_b7 + eps_b7)  # Standardize the signal.

print("normalized value:", round(float(normed_b7), 3))  # Inspect the z-like value.

assert round(float(normed_b7), 3) == 4.9  # Verify the lesson number.

▶ What you'll see: the normalized score is about `4.900`.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a scale plot.
plt.bar(["mean", "score", "normalized"], [mean_b7, score_b7, normed_b7], color=["gray", "steelblue", "purple"])  # Compare raw and normalized quantities.
plt.title("Basic 7: normalization changes scale")  # Title the chart.
plt.show()  # Display the plot.

▶ What you'll see: the normalized value is much larger because the variance scale is small.

👀 Takeaway: normalization says how unusual a signal is relative to its local batch or feature scale.

### Basic 8 — Use stable softmax

**Goal.** Turn two scores into a probability without unstable exponentials, because deep learning usually trains from comparisons. We build it in 2 steps.

In [ ]:
logits_b8 = np.array([3.45, 0.4], dtype=np.float32)  # Compare the lesson score with a baseline.
probs_b8 = softmax_stable(logits_b8)  # Compute max-shifted softmax probabilities.

print("probabilities:", np.round(probs_b8, 3))  # Inspect the calibrated comparison.

assert round(float(probs_b8[0]), 3) == 0.955  # Verify the lesson probability.

▶ What you'll see: the higher score receives probability about `0.955`.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a probability chart.
plt.bar(["score", "baseline"], probs_b8, color=["steelblue", "gray"])  # Plot softmax probabilities.
plt.title("Basic 8: softmax comparison")  # Title the chart.
plt.ylabel("probability")  # Label the y-axis.
plt.show()  # Display the plot.

▶ What you'll see: almost all probability mass goes to the larger score.

👀 Takeaway: softmax depends on score differences, so subtracting the max preserves the result and improves numerical safety.

### Basic 9 — Unscale a stored gradient

**Goal.** Scale and unscale a gradient, because mixed precision protects tiny gradients before low-precision storage. We build it in 2 steps.

In [ ]:
g_b9 = np.array([2.1, 1e-8], dtype=np.float32)  # Store one ordinary and one tiny gradient.
scale_b9 = 4096.0  # Choose a power-of-two loss scale.
stored_b9 = fp16(g_b9 * scale_b9)  # Store scaled gradients in FP16.
unscaled_b9 = stored_b9 / scale_b9  # Unscale after storage.

print("stored scaled:", stored_b9)  # Inspect scaled low-precision values.
print("unscaled:", unscaled_b9)  # Inspect recovered gradients.

assert unscaled_b9[1] > 0.0  # Verify that the tiny gradient survived.

▶ What you'll see: the scaled tiny gradient is representable and becomes nonzero after unscaling.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a tiny-gradient comparison.
plt.bar(["true", "unscaled"], [g_b9[1], unscaled_b9[1]], color=["gray", "seagreen"])  # Compare original and recovered tiny gradient.
plt.title("Basic 9: scaled gradient survives")  # Title the chart.
plt.ylabel("gradient")  # Label the value axis.
plt.show()  # Display the plot.

▶ What you'll see: the recovered value is close to the tiny original instead of exactly zero.

👀 Takeaway: loss scaling changes the intermediate representation, not the intended update direction.

### Basic 10 — Keep a FP32 master parameter

**Goal.** Update a parameter from an unscaled gradient while keeping the accumulator in FP32, because small steps must add up over time. We build it in 2 steps.

In [ ]:
theta_b10 = np.float32(2.0)  # Store the master parameter in float32.
g16_b10 = fp16(np.array([2.1], dtype=np.float32))[0]  # Simulate a gradient stored in FP16.
eta_b10 = np.float32(0.09)  # Store learning rate.
theta_new_b10 = theta_b10 - eta_b10 * g16_b10  # Apply the mixed-precision update to the FP32 master.

print("stored gradient:", float(g16_b10))  # Inspect rounded gradient.
print("new master theta:", round(float(theta_new_b10), 3))  # Inspect updated master parameter.

assert round(float(theta_new_b10), 3) == 1.811  # Verify the lesson update survives.

▶ What you'll see: the FP32 master update still lands at about `1.811`.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a before/after chart.
plt.bar(["θ32 before", "θ32 after"], [theta_b10, theta_new_b10], color=["gray", "seagreen"])  # Show master-copy movement.
plt.title("Basic 10: mixed-precision master update")  # Title the chart.
plt.ylabel("parameter value")  # Label the value axis.
plt.show()  # Display the plot.

▶ What you'll see: the master parameter moves by the scaled gradient descent step.

👀 Takeaway: mixed precision usually stores big tensors cheaply while preserving optimization state in higher precision.

## 🟡 Easy

### Easy 1 — Compare format error across scales

**Goal.** Sweep magnitudes and compare FP16 with bfloat16, because a format's useful precision depends on both mantissa and exponent range. We build it in 3 steps.

In [ ]:
vals_e1 = np.array([1e-5, 1e-3, 1.0, 3.45, 100.0, 1e5], dtype=np.float32)  # Sweep small to large values.
f16_e1 = fp16(vals_e1)  # Round through FP16.
bf_e1 = bf16(vals_e1)  # Round through simulated bfloat16.

print("fp16:", f16_e1)  # Inspect FP16 rounded values.
print("bf16:", bf_e1)  # Inspect bfloat16 rounded values.

assert np.isinf(f16_e1[-1]) and np.isfinite(bf_e1[-1])  # Check the large-range difference.

▶ What you'll see: FP16 overflows the largest value, while bfloat16 keeps it finite.

In [ ]:
safe_f16_e1 = np.where(np.isfinite(f16_e1), f16_e1, 65504.0)  # Replace inf with FP16's max for plotting.
err_f16_e1 = np.abs(safe_f16_e1 - vals_e1)  # Compute displayable FP16 error.
err_bf_e1 = np.abs(bf_e1 - vals_e1)  # Compute bfloat16 error.

print("error at 3.45 fp16/bf16:", round(float(err_f16_e1[3]), 5), round(float(err_bf_e1[3]), 5))  # Inspect a normal-scale value.

▶ What you'll see: FP16 is more precise around `3.45`, while bfloat16 is more robust at extreme range.

In [ ]:
plt.figure(figsize=(5, 3))  # Create an error comparison plot.
plt.loglog(vals_e1, err_f16_e1 + 1e-12, marker="o", label="fp16")  # Plot FP16 error.
plt.loglog(vals_e1, err_bf_e1 + 1e-12, marker="o", label="bf16")  # Plot bfloat16 error.
plt.title("Easy 1: format error by magnitude")  # Title the figure.
plt.xlabel("value magnitude")  # Label x-axis.
plt.ylabel("absolute error")  # Label y-axis.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: error curves cross because the two formats make different precision-versus-range tradeoffs.

👀 Takeaway: choosing FP16 or bfloat16 is a numerical design choice, not just a memory choice.

### Easy 2 — Show underflow and loss scaling

**Goal.** Demonstrate that loss scaling can preserve tiny gradients that direct FP16 storage erases. We build it in 3 steps.

In [ ]:
grads_e2 = np.array([1e-4, 1e-6, 1e-8, 1e-10], dtype=np.float32)  # Create gradients over several tiny scales.
direct_e2 = fp16(grads_e2)  # Store directly in FP16.
scale_e2 = 8192.0  # Choose a large but safe scale.
scaled_e2 = fp16(grads_e2 * scale_e2) / scale_e2  # Scale, store, and unscale.

print("direct:", direct_e2)  # Inspect direct storage.
print("scaled:", scaled_e2)  # Inspect recovered storage.

assert direct_e2[2] == 0.0 and scaled_e2[2] > 0.0  # Verify the 1e-8 rescue.

▶ What you'll see: `1e-8` underflows directly but survives the scale-and-unscale path.

In [ ]:
survived_direct_e2 = direct_e2 > 0  # Mark which gradients survived direct FP16 storage.
survived_scaled_e2 = scaled_e2 > 0  # Mark which gradients survived scaled FP16 storage.

print("survived direct:", survived_direct_e2.astype(int))  # Inspect survival mask.
print("survived scaled:", survived_scaled_e2.astype(int))  # Inspect survival mask.

▶ What you'll see: scaling increases the count of nonzero stored gradients.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a survival chart.
plt.semilogy(grads_e2, direct_e2 + 1e-12, marker="o", label="direct fp16")  # Plot direct values.
plt.semilogy(grads_e2, scaled_e2 + 1e-12, marker="o", label="scaled/unscaled")  # Plot recovered values.
plt.title("Easy 2: underflow protection")  # Title the plot.
plt.xlabel("true gradient")  # Label x-axis.
plt.ylabel("stored then recovered")  # Label y-axis.
plt.legend()  # Show labels.
plt.show()  # Display the chart.

▶ What you'll see: the scaled curve stays nonzero farther into the tiny-gradient region.

👀 Takeaway: loss scaling keeps weak gradient signal numerically visible until the master update can use it.

### Easy 3 — Detect overflow before unscale

**Goal.** Show why a scaler must skip or reduce scale after overflow, because an infinite stored gradient cannot be rescued by unscaling. We build it in 3 steps.

In [ ]:
grads_e3 = np.array([2.1, 20.0, 200.0], dtype=np.float32)  # Use increasingly large gradients.
scale_e3 = 1024.0  # Choose a scale that is too high for the largest gradient.
stored_e3 = fp16(grads_e3 * scale_e3)  # Store scaled gradients in FP16.
finite_e3 = np.isfinite(stored_e3)  # Check whether scaling overflowed.

print("stored scaled gradients:", stored_e3)  # Inspect scaled storage.
print("finite mask:", finite_e3.astype(int))  # Inspect overflow detection.

assert not finite_e3[-1]  # Verify the largest scaled gradient overflowed.

▶ What you'll see: the largest scaled gradient becomes `inf`, so the step is unsafe.

In [ ]:
new_scale_e3 = scale_e3 / 2 if not np.all(finite_e3) else scale_e3  # Reduce scale when overflow is detected.
stored_retry_e3 = fp16(grads_e3 * new_scale_e3)  # Retry at the smaller scale.

print("new scale:", new_scale_e3)  # Inspect the reduced scale.
print("retry finite:", np.isfinite(stored_retry_e3).astype(int))  # Inspect retry status.

▶ What you'll see: reducing the scale is the control response to overflow.

In [ ]:
plt.figure(figsize=(5, 3))  # Create an overflow status chart.
plt.bar(["g=2.1", "g=20", "g=200"], finite_e3.astype(float), color=["seagreen" if v else "crimson" for v in finite_e3])  # Plot finite flags.
plt.title("Easy 3: overflow detection")  # Title the chart.
plt.ylabel("1 = finite after scaling")  # Label status axis.
plt.show()  # Display the plot.

▶ What you'll see: overflow is a binary safety check before an optimizer trusts the step.

👀 Takeaway: dynamic loss scaling raises scale to avoid underflow but lowers it when overflow appears.

### Easy 4 — Accumulate many small updates

**Goal.** Compare FP32 master accumulation with FP16-only subtraction, because tiny steps can vanish at the parameter's current spacing. We build it in 3 steps.

In [ ]:
steps_e4 = 40  # Use enough updates to show accumulation.
step_e4 = np.float32(1e-4)  # Choose a small gradient-descent movement.
master_e4 = np.float32(1.0)  # Store a FP32 master parameter.
low_e4 = np.float16(1.0)  # Store a FP16-only parameter.
trace_master_e4 = []  # Track master values.
trace_low_e4 = []  # Track low-precision-only values.

print("initial spacing near 1 for fp16:", float(np.nextafter(np.float16(1.0), np.float16(2.0)) - np.float16(1.0)))  # Inspect local FP16 spacing.

▶ What you'll see: the FP16 spacing near 1 is about `0.0009766`, much larger than one `1e-4` step.

In [ ]:
for _ in range(steps_e4):  # Apply repeated small steps.
    master_e4 = np.float32(master_e4 - step_e4)  # Update in FP32.
    low_e4 = np.float16(low_e4 - np.float16(step_e4))  # Update in FP16.
    trace_master_e4.append(float(master_e4))  # Save FP32 path.
    trace_low_e4.append(float(low_e4))  # Save FP16 path.

print("final master/low:", round(trace_master_e4[-1], 6), trace_low_e4[-1])  # Inspect final parameters.

assert round(trace_master_e4[-1], 3) == 0.996  # Verify FP32 accumulated 40 steps.

▶ What you'll see: the FP32 master moves by about `0.004`, while the FP16 path is quantized.

In [ ]:
plt.figure(figsize=(5, 3))  # Create an accumulation plot.
plt.plot(trace_master_e4, label="fp32 master", color="seagreen")  # Plot precise accumulation.
plt.plot(trace_low_e4, label="fp16-only", color="crimson")  # Plot rounded updates.
plt.title("Easy 4: small update accumulation")  # Title the chart.
plt.xlabel("step")  # Label x-axis.
plt.ylabel("parameter")  # Label y-axis.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: the master curve changes smoothly; the FP16-only curve changes in coarse jumps or not at all.

👀 Takeaway: optimizer state and master weights need enough precision to accumulate many weak learning signals.

### Easy 5 — Estimate memory for a stack of layers

**Goal.** Scale the lesson's activation-memory calculation across depth, because memory pressure grows across stored layers and batches. We build it in 3 steps.

In [ ]:
layers_e5 = np.array([1, 8, 32, 96])  # Choose numbers of stored activation blocks.
kb32_e5 = layers_e5 * activation_kb(4, 128, 4)  # Compute float32 memory for each depth.
kb16_e5 = layers_e5 * activation_kb(4, 128, 2)  # Compute 16-bit memory for each depth.

print("float32 KB:", kb32_e5)  # Inspect 32-bit memory growth.
print("16-bit KB:", kb16_e5)  # Inspect 16-bit memory growth.

assert kb32_e5[0] == 2.0 and kb16_e5[-1] == 96.0  # Verify concrete memory values.

▶ What you'll see: each extra stored block adds `2 KB` in float32 and `1 KB` in 16-bit storage.

In [ ]:
saved_e5 = kb32_e5 - kb16_e5  # Compute memory saved by 16-bit activations.

print("saved KB:", saved_e5)  # Inspect absolute savings by depth.

▶ What you'll see: saved memory grows linearly with the number of stored activation blocks.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a depth-memory plot.
plt.plot(layers_e5, kb32_e5, marker="o", label="float32")  # Plot 32-bit memory.
plt.plot(layers_e5, kb16_e5, marker="o", label="16-bit")  # Plot 16-bit memory.
plt.fill_between(layers_e5, kb16_e5, kb32_e5, alpha=0.2, color="seagreen")  # Highlight savings.
plt.title("Easy 5: activation memory by depth")  # Title the figure.
plt.xlabel("stored blocks")  # Label depth axis.
plt.ylabel("KB")  # Label memory axis.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: the green shaded region is memory saved by using 2-byte activation storage.

👀 Takeaway: mixed precision matters most where large tensors are stored or moved repeatedly.

## 🔴 Advanced

### Advanced 1 — Simulate a full mixed-precision scalar update

**Goal.** Put scaling, FP16 gradient storage, unscaling, and a FP32 master parameter into one update, because that is the core mixed-precision training rule. We build it in 4 steps.

In [ ]:
theta_a1 = np.float32(2.0)  # Keep the master parameter in FP32.
grad_true_a1 = np.float32(2.1)  # Use the lesson gradient.
eta_a1 = np.float32(0.09)  # Use the lesson learning rate.
scale_a1 = np.float32(512.0)  # Choose a scale that does not overflow this gradient.

print("master θ:", theta_a1, "true gradient:", grad_true_a1)  # Inspect update ingredients.

▶ What you'll see: the update begins from `θ=2.0` and gradient `2.1`.

In [ ]:
grad16_scaled_a1 = fp16(np.array([grad_true_a1 * scale_a1], dtype=np.float32))[0]  # Store scaled gradient in FP16.
grad_unscaled_a1 = grad16_scaled_a1 / scale_a1  # Unscale for the optimizer.

print("scaled stored gradient:", float(grad16_scaled_a1))  # Inspect low-precision storage.
print("unscaled gradient:", float(grad_unscaled_a1))  # Inspect recovered gradient.

▶ What you'll see: the scaled gradient is large enough for FP16 storage and returns close to `2.1` after unscaling.

In [ ]:
theta_new_a1 = theta_a1 - eta_a1 * grad_unscaled_a1  # Apply θ32 ← θ32 − η unscale(g16).

print("updated θ32:", round(float(theta_new_a1), 3))  # Inspect the mixed-precision update result.

assert round(float(theta_new_a1), 3) == 1.811  # Verify the canonical lesson update.

▶ What you'll see: the mixed-precision pipeline reproduces the expected `1.811` parameter value.

In [ ]:
plt.figure(figsize=(5, 3))  # Create an update-pipeline chart.
plt.bar(["θ before", "η·g", "θ after"], [theta_a1, eta_a1 * grad_unscaled_a1, theta_new_a1], color=["gray", "orange", "seagreen"])  # Show before, step, and after.
plt.title("Advanced 1: mixed-precision update")  # Title the chart.
plt.ylabel("value")  # Label value axis.
plt.show()  # Display the plot.

▶ What you'll see: the update bar is the small subtraction that turns `2.0` into about `1.811`.

👀 Takeaway: the central rule is low-precision gradient storage plus high-precision parameter accumulation.

### Advanced 2 — Choose a safe loss scale window

**Goal.** Sweep scale factors and find which ones avoid both underflow and overflow for a gradient batch. We build it in 4 steps.

In [ ]:
grads_a2 = np.array([2.1, 1e-4, 1e-8, 200.0], dtype=np.float32)  # Mix ordinary, tiny, and large gradients.
scales_a2 = np.array([1, 128, 4096, 65536], dtype=np.float32)  # Try increasingly aggressive loss scales.

print("gradients:", grads_a2)  # Inspect the gradient batch.
print("scales:", scales_a2)  # Inspect candidate scales.

▶ What you'll see: the same gradient batch contains both underflow and overflow risks.

In [ ]:
nonzero_counts_a2 = []  # Count how many gradients remain nonzero after storage.
finite_flags_a2 = []  # Track whether all scaled gradients are finite.
for s_a2 in scales_a2:  # Test each candidate scale.
    stored_a2 = fp16(grads_a2 * s_a2)  # Store scaled gradients in FP16.
    recovered_a2 = stored_a2 / s_a2  # Unscale for inspection.
    nonzero_counts_a2.append(int(np.sum(recovered_a2 != 0)))  # Count nonzero recovered gradients.
    finite_flags_a2.append(bool(np.all(np.isfinite(stored_a2))))  # Record overflow safety.

print("nonzero counts:", nonzero_counts_a2)  # Inspect underflow protection.
print("all finite:", finite_flags_a2)  # Inspect overflow protection.

assert nonzero_counts_a2[0] < nonzero_counts_a2[2] and finite_flags_a2[-1] == False  # Verify the tradeoff.

▶ What you'll see: larger scales recover more tiny gradients until they overflow large gradients.

In [ ]:
safe_a2 = np.array(finite_flags_a2)  # Convert safety flags to an array.
useful_a2 = np.array(nonzero_counts_a2)  # Convert nonzero counts to an array.
best_idx_a2 = int(np.argmax(np.where(safe_a2, useful_a2, -1)))  # Choose the safe scale with most nonzero gradients.

print("chosen scale:", float(scales_a2[best_idx_a2]))  # Inspect the selected scale.

▶ What you'll see: the best scale is the largest useful one before overflow dominates.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a scale sweep plot.
plt.semilogx(scales_a2, nonzero_counts_a2, marker="o", label="nonzero recovered")  # Plot gradient survival.
plt.semilogx(scales_a2, np.array(finite_flags_a2, dtype=int) * len(grads_a2), marker="s", label="finite status × count")  # Plot overflow status on same scale.
plt.title("Advanced 2: loss-scale window")  # Title the figure.
plt.xlabel("loss scale")  # Label scale axis.
plt.ylabel("count")  # Label count axis.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: there is a usable middle region: enough scale to avoid underflow, not so much that overflow appears.

👀 Takeaway: dynamic scaling is a feedback controller for the representable gradient range.

### Advanced 3 — Compare accumulation error in a dot product

**Goal.** Simulate low-precision dot-product inputs with float32 accumulation, because matrix units often multiply low-precision operands but accumulate into a wider type. We build it in 4 steps.

In [ ]:
rng_a3 = np.random.default_rng(3)  # Use a local generator for reproducible vectors.
x_a3 = rng_a3.normal(size=256).astype(np.float32)  # Create one activation vector.
w_a3 = rng_a3.normal(size=256).astype(np.float32)  # Create one weight vector.
true_dot_a3 = float(x_a3 @ w_a3)  # Compute the float32 reference dot product.

print("float32 dot:", round(true_dot_a3, 3))  # Inspect the reference.

▶ What you'll see: a reference dot product from full-precision inputs.

In [ ]:
dot_fp16in_a3 = float(fp16(x_a3) @ fp16(w_a3))  # Round operands through FP16, accumulate in NumPy float32.
dot_bf16in_a3 = float(bf16(x_a3) @ bf16(w_a3))  # Round operands through bfloat16, accumulate in NumPy float32.
err_fp16_a3 = abs(dot_fp16in_a3 - true_dot_a3)  # Compute FP16-input error.
err_bf16_a3 = abs(dot_bf16in_a3 - true_dot_a3)  # Compute bfloat16-input error.

print("fp16-input dot:", round(dot_fp16in_a3, 3), "error:", round(err_fp16_a3, 4))  # Inspect FP16-input result.
print("bf16-input dot:", round(dot_bf16in_a3, 3), "error:", round(err_bf16_a3, 4))  # Inspect bfloat16-input result.

assert err_fp16_a3 < 0.05  # Verify the low-precision operand error is small for this scale.

▶ What you'll see: operand rounding changes the dot product slightly even when accumulation is wider.

In [ ]:
contrib_err_a3 = (fp16(x_a3) * fp16(w_a3)) - (x_a3 * w_a3)  # Compute per-coordinate contribution error.

print("mean contribution error:", round(float(np.mean(contrib_err_a3)), 6))  # Inspect average local error.

▶ What you'll see: many small contribution errors combine into the final dot-product difference.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a histogram of contribution errors.
plt.hist(contrib_err_a3, bins=30, color="steelblue", edgecolor="white")  # Plot distribution of local errors.
plt.title("Advanced 3: dot-product contribution error")  # Title the plot.
plt.xlabel("low-precision contribution − float32 contribution")  # Label x-axis.
plt.ylabel("count")  # Label y-axis.
plt.show()  # Display the histogram.

▶ What you'll see: most contribution errors cluster near zero, which is why wider accumulation can keep the dot product usable.

👀 Takeaway: tensor-core-style arithmetic depends on low-precision storage being paired with careful accumulation.

### Advanced 4 — Show why stable softmax matters more in low precision

**Goal.** Compare naive and max-shifted softmax on large logits, because exponentials can overflow before probabilities are normalized. We build it in 4 steps.

In [ ]:
logits_a4 = np.array([90.0, 87.0, 80.0], dtype=np.float32)  # Choose logits large enough to stress exponentials.
with np.errstate(over="ignore", invalid="ignore"):
    naive_exp_a4 = np.exp(logits_a4)  # Compute naive exponentials.

print("naive exp finite:", np.isfinite(naive_exp_a4).astype(int))  # Inspect overflow status.

assert not np.all(np.isfinite(naive_exp_a4))  # Verify naive exponentials overflow in float32.

▶ What you'll see: at least one naive exponential is not finite.

In [ ]:
stable_probs_a4 = softmax_stable(logits_a4)  # Compute stable softmax by shifting logits first.

print("stable probabilities:", np.round(stable_probs_a4, 4))  # Inspect finite probabilities.
print("sum:", round(float(np.sum(stable_probs_a4)), 6))  # Verify probabilities sum to one.

assert round(float(np.sum(stable_probs_a4)), 6) == 1.0  # Check normalization.

▶ What you'll see: stable softmax returns finite probabilities that sum to exactly one up to rounding.

In [ ]:
shifted_a4 = logits_a4 - np.max(logits_a4)  # Compute the numerically safe logits.

print("shifted logits:", shifted_a4)  # Inspect the shifted values sent to exp.

▶ What you'll see: the largest shifted logit is zero, so its exponential is safe and equal to one.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a probability chart.
plt.bar(["class 0", "class 1", "class 2"], stable_probs_a4, color="purple")  # Plot stable softmax probabilities.
plt.title("Advanced 4: stable softmax probabilities")  # Title the chart.
plt.ylabel("probability")  # Label y-axis.
plt.show()  # Display the plot.

▶ What you'll see: probabilities are sharply peaked but finite.

👀 Takeaway: algebraically equivalent stable formulas are part of mixed-precision engineering, not optional cleanup.

### Advanced 5 — Estimate arithmetic intensity

**Goal.** Compare compute work with bytes moved for a matrix multiply, because hardware speed depends on keeping compute units fed rather than just counting FLOPs. We build it in 4 steps.

In [ ]:
m_a5, k_a5, n_a5 = 64, 128, 64  # Define a small matrix multiply shape: (m×k) @ (k×n).
flops_a5 = 2 * m_a5 * k_a5 * n_a5  # Count multiply and add as two floating-point operations.
bytes32_a5 = (m_a5 * k_a5 + k_a5 * n_a5 + m_a5 * n_a5) * 4  # Read A and B, write C in float32.
bytes16_a5 = (m_a5 * k_a5 + k_a5 * n_a5) * 2 + (m_a5 * n_a5) * 4  # Use 16-bit inputs and float32 output.

print("MFLOPs:", flops_a5 / 1e6)  # Inspect compute work.
print("bytes float32/16input:", bytes32_a5, bytes16_a5)  # Inspect memory traffic estimate.

assert round(flops_a5 / 1e6, 3) == 1.049  # Verify concrete FLOP count.

▶ What you'll see: the matrix multiply performs about `1.049` million FLOPs with substantial input/output traffic.

In [ ]:
intensity32_a5 = flops_a5 / bytes32_a5  # Compute FLOPs per byte for float32 tensors.
intensity16_a5 = flops_a5 / bytes16_a5  # Compute FLOPs per byte for 16-bit inputs.

print("FLOPs/byte float32:", round(float(intensity32_a5), 3))  # Inspect arithmetic intensity.
print("FLOPs/byte 16-input:", round(float(intensity16_a5), 3))  # Inspect improved intensity.

assert intensity16_a5 > intensity32_a5  # Verify 16-bit inputs increase work per byte moved.

▶ What you'll see: reducing input bytes increases arithmetic intensity.

In [ ]:
labels_a5 = ["float32 tensors", "16-bit inputs"]  # Name the two memory cases.
bytes_kb_a5 = np.array([bytes32_a5, bytes16_a5]) / 1024  # Convert bytes to KB for plotting.

print("traffic KB:", np.round(bytes_kb_a5, 1))  # Inspect memory traffic in KB.

▶ What you'll see: the 16-bit-input estimate moves fewer kilobytes for the same multiply.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a hardware-efficiency comparison.
plt.bar(labels_a5, [intensity32_a5, intensity16_a5], color=["gray", "seagreen"])  # Plot FLOPs per byte.
plt.title("Advanced 5: arithmetic intensity")  # Title the figure.
plt.ylabel("FLOPs per byte moved")  # Label the efficiency axis.
plt.xticks(rotation=10)  # Rotate labels for readability.
plt.show()  # Display the chart.

▶ What you'll see: the 16-bit-input bar is higher, meaning the same compute needs less memory movement.

👀 Takeaway: mixed precision can speed training because it improves both tensor-core compute paths and memory bandwidth pressure.